In [17]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
from PIL import Image
import os
import json

In [18]:
class MVTecDataset(Dataset):
    def __init__(self, samples, root_dir, transform=None):
        self.samples = samples
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.samples[idx]['filepath'])
        image = Image.open(img_name).convert("RGB")
        label = self.samples[idx]['defect']['label']
        if self.transform:
            image = self.transform(image)
        return image, label

In [19]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

In [20]:
ROOT_DIR = "D:\\Projects\\Industrial Defect Detection\\dataset\\mvtec-ad"

with open(os.path.join(ROOT_DIR, "samples.json"), "r") as f:
    data = json.load(f)

samples = data['samples']

In [21]:
training_dataset = MVTecDataset(samples, ROOT_DIR, transform=transform)
testing_dataset = MVTecDataset(samples, ROOT_DIR, transform=transform)

In [22]:
print("Training data length: ", len(training_dataset))
print("Testing data length: ", len(testing_dataset))

image, label = training_dataset[0]

print("Image shape: ", image.shape)
print("Label: ", label)

Training data length:  5354
Testing data length:  5354
Image shape:  torch.Size([3, 128, 128])
Label:  good


In [24]:
train_samples = [
    sample for sample in samples
    if sample["category"]["label"] == "metal_nut"
    and sample["defect"]["label"] == "good"
    and sample["split"] == "train"
]

test_samples = [
    sample for sample in samples
    if sample["category"]["label"] == "metal_nut"
    and sample["split"] == "test"
]

In [25]:
print("Training samples length: ", len(train_samples))
print("Testing samples length: ", len(test_samples))

Training samples length:  220
Testing samples length:  115


In [28]:
train_loader = DataLoader(training_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(testing_dataset, batch_size=16, shuffle=False)

In [29]:
images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Number of labels:", len(labels))

Batch image shape: torch.Size([16, 3, 128, 128])
Number of labels: 16
